In [2]:
import numpy as np
from astropy.cosmology import Planck18 as cosmo
from astropy import units as u

def compute_luminosity(redshift, apparent_mag, M_sun=4.83, L_sun=3.828e33):
	"""
	Compute luminosity in erg/s from redshift and apparent magnitude.

	Parameters:
		redshift (float or array): Redshift of the object.
		apparent_mag (float or array): Apparent magnitude (e.g., in V band).
		M_sun (float): Absolute magnitude of the Sun in the same band (default V = 4.83).
		L_sun (float): Luminosity of the Sun in erg/s.

	Returns:
		luminosity (float or array): Luminosity in erg/s.
	"""
	# Compute luminosity distance in parsecs
	D_L = cosmo.luminosity_distance(redshift).to(u.pc).value
	
	# Compute absolute magnitude
	M_abs = apparent_mag - 5 * (np.log10(D_L) - 1)
	
	# Convert absolute magnitude to luminosity
	luminosity = L_sun * 10**(-0.4 * (M_abs - M_sun))
	return luminosity, M_abs


In [3]:
apparent_mag_g = 16.82
apparent_mag_r = 16.79
redshift = 0.034


print("Luminosity r band: {0:2e} erg/s, corresponding to abs magnitude: {1:.2f}".format(*compute_luminosity(redshift, apparent_mag_r)))
print("Luminosity g band: {0:2e} erg/s, corresponding to abs magnitude: {1:.2f}".format(*compute_luminosity(redshift, apparent_mag_g)))

Luminosity r band: 1.503084e+43 erg/s, corresponding to abs magnitude: -19.16
Luminosity g band: 1.462121e+43 erg/s, corresponding to abs magnitude: -19.13


In [6]:
compute_luminosity(redshift, apparent_mag_r)

(3.991967601162225e+44, -22.71553769696923)

In [33]:
from astropy.io import fits
from astropy.table import Table
import numpy as np

# Input / output files
fits_file = "/home/lmiguel/Data/TDE_spectra/2022yhf - ANU/OBK-1089088-WiFeS-Splice-UT20250719T125901-9.p12.fits"
ascii_file = fits_file[:-5] +".dat"



# Read FITS table (usually extension 1)
try:
    table = Table.read(fits_file, hdu=1)
except ValueError:
    print("Not standard Table in the file. Trying in a different way.")
    # Based on the format given once by SSO-ANU team
    table = Table(fits.open(fits_file)[1].data)
    hdul = fits.open(fits_file)
    hdr = hdul[0].header

    flux = hdul[0].data
    variance = hdul[1].data

    n = len(flux)
    crval = hdr["CRVAL1"]
    cdelt = hdr["CDELT1"]
    crpix = hdr.get("CRPIX1", 1)

    wavelength = crval + (np.arange(n) + 1 - crpix) * cdelt

    table = Table()
    table["wavelength"] = wavelength
    table["flux"] = flux
    table["variance"] = variance

# Write to ASCII
table.write(ascii_file, format="ascii", overwrite=True)

Not standard Table in the file. Trying in a different way.


In [31]:
ascii_file

'/home/lmiguel/Data/TDE_spectra/2025qhf - ANU/OBK-1096448-WiFeS-Splice-UT20250723T115937-6.p12_SN.dat'